# AgentSecEval — Exploratory Analysis

This notebook loads `results/metrics_summary.csv` and produces the key visualisations for the research questions.

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path('../results')
CSV_PATH = RESULTS_DIR / 'metrics_summary.csv'

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} run records')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Cell 2: ASR heatmap (scenario × isolation mode)
pivot = df.groupby(['scenario_id', 'isolation_mode'])['asr'].mean().unstack(fill_value=0) * 100

fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns) * 2), max(4, len(pivot) * 0.6)))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=100)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Attack Success Rate (%) — Scenario × Isolation Mode')
ax.set_xlabel('Isolation Mode')
ax.set_ylabel('Scenario ID')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9,
                color='white' if val > 60 else 'black')

plt.colorbar(im, ax=ax, label='ASR (%)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'asr_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Cell 3: Breach rates bar chart by isolation mode
breach_df = df.groupby('isolation_mode')[['fs_breach', 'net_breach']].mean() * 100

x = np.arange(len(breach_df))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, breach_df['fs_breach'], width, label='FS Breach Rate', color='steelblue')
bars2 = ax.bar(x + width/2, breach_df['net_breach'], width, label='Net Breach Rate', color='tomato')

ax.set_xlabel('Isolation Mode')
ax.set_ylabel('Breach Rate (%)')
ax.set_title('RQ1: Filesystem and Network Breach Rates by Isolation Mode')
ax.set_xticks(x)
ax.set_xticklabels(breach_df.index)
ax.set_ylim(0, 105)
ax.legend()
ax.bar_label(bars1, fmt='%.0f%%', padding=3)
ax.bar_label(bars2, fmt='%.0f%%', padding=3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'breach_rates.png', dpi=150)
plt.show()

In [ ]:
# Cell 4: RQ2 finding — ASR comparison across modes for agent-layer attacks
agent_layer_categories = [
    'prompt_injection_direct',
    'prompt_injection_indirect',
    'tool_abuse',
    'data_exfiltration',
    'memory_attack',
]

agent_df = df[df['category'].isin(agent_layer_categories)]
rq2 = agent_df.groupby(['isolation_mode'])['asr'].agg(['mean', 'std']) * 100
rq2.columns = ['ASR Mean (%)', 'ASR Std (%)']
rq2 = rq2.round(2)

print('RQ2 Finding: ASR for agent-layer attacks by isolation mode')
print('=' * 55)
print(rq2.to_string())
print()
print('Interpretation:')
print('  If ASR values are statistically similar across modes,')
print('  isolation does NOT mitigate agent-layer attack categories.')
print('  These attacks must be addressed at the application/model level.')